# 🔍 Nemotron Loss Debug Notebook

**目的**: 诊断 `sft_thinking.csv` 9000样本训练时 loss 异常大的原因

**诊断项**:
1. Token 长度分布 & 截断分析
2. 初始 loss (未训任何步) — 基线参考
3. Per-token loss 分析: prompt / thinking / answer 各段贡献
4. 快速训练对比: thinking vs answer-only × alpha 16 vs 64

**环境**: Colab Pro (A100), 模型用 kagglehub 下载 + 4-bit QLoRA

> 4-bit 量化仅用于调试 loss 趋势，绝对值会与 BF16 略有差异，但相对趋势一致

In [ ]:
%%capture
!pip install -U datasets trl peft accelerate bitsandbytes sentencepiece protobuf kagglehub wandb
!pip install mamba-ssm causal-conv1d --no-build-isolation

In [ ]:
import os, sys, gc, re, json, math, time
import numpy as np
import pandas as pd

# =============================================
#  禁用 wandb (避免版本冲突, 我们用 report_to='none')
# =============================================
os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "disabled"

import torch
import torch.nn.functional as F
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig

# =============================================
#  Kaggle API 认证 (kagglehub 需要)
# =============================================
os.environ["KAGGLE_USERNAME"] = "hastws"
os.environ["KAGGLE_KEY"] = "319f973e08a55f1c92601009be770909"

print(f"torch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    mem_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU memory: {mem_gb:.1f} GB")

print(f"Kaggle user: {os.environ['KAGGLE_USERNAME']}")

In [ ]:
# =============================================
#  Google Drive 挂载 & 数据路径
# =============================================
from google.colab import drive
drive.mount('/content/drive')

DRIVE_PROJECT = '/content/drive/MyDrive/nemotron_competition'
DATA_DIR = os.path.join(DRIVE_PROJECT, 'data')
COMP_DATA = os.path.join(DRIVE_PROJECT, 'competition_data')

# 检查数据文件
for fname, fdir in [('sft_thinking.csv', DATA_DIR), ('train.csv', COMP_DATA)]:
    path = os.path.join(fdir, fname)
    if os.path.exists(path):
        size = os.path.getsize(path) / 1024 / 1024
        print(f"  ✅ {fname} ({size:.1f} MB)")
    else:
        print(f"  ❌ {fname} NOT FOUND at {path}")
        print(f"     请上传到 Google Drive: {fdir}/")

# =============================================
#  模型下载 (kagglehub — 与 Kaggle notebook 一致)
# =============================================
import kagglehub

print("\n下载 Nemotron 模型 (首次下载约 60GB, 之后缓存)...")
try:
    MODEL_PATH = kagglehub.model_download(
        "metric/nemotron-3-nano-30b-a3b-bf16/transformers/default"
    )
    print(f"✅ 模型路径: {MODEL_PATH}")
except Exception as e:
    print(f"❌ kagglehub 下载失败: {e}")
    print("   请先运行: kaggle config set -n username -v YOUR_USERNAME")
    print("              kaggle config set -n key -v YOUR_API_KEY")
    print("   或设置环境变量 KAGGLE_USERNAME / KAGGLE_KEY")
    raise

## 1. 数据分析 — 了解 loss 来源

In [ ]:
# =============================================
#  加载训练数据
# =============================================
train_csv_path = os.path.join(DATA_DIR, 'sft_thinking.csv')
if not os.path.exists(train_csv_path):
    # fallback: 用原始 train.csv
    train_csv_path = os.path.join(COMP_DATA, 'train.csv')
    print(f"⚠️ sft_thinking.csv 不存在，使用 train.csv")

df = pd.read_csv(train_csv_path)
print(f"Loaded: {len(df)} rows from {os.path.basename(train_csv_path)}")
print(f"Columns: {list(df.columns)}")

# 补全缺失列
if 'type' not in df.columns:
    def infer_type(p):
        p = p.lower()
        if 'bit manipulation' in p or '8-bit binary' in p: return 'bit_ops'
        if 'numeral system' in p: return 'numeral'
        if 'encrypt' in p or 'decrypt' in p: return 'cipher'
        if 'gravitational' in p or 'gravity' in p or 'free-fall' in p: return 'gravity'
        if 'unit' in p and ('convert' in p or 'conversion' in p): return 'unit_conv'
        if 'symbol' in p or 'transformation rule' in p: return 'symbol'
        return 'unknown'
    df['type'] = df['prompt'].apply(infer_type)

if 'thinking' not in df.columns:
    df['thinking'] = ''

# 数据统计
has_thinking = df['thinking'].fillna('').str.strip().str.len() > 0
print(f"\n{'='*60}")
print(f"  数据统计")
print(f"{'='*60}")
print(f"  总行数: {len(df)}")
print(f"  有 thinking: {has_thinking.sum()} ({has_thinking.mean()*100:.1f}%)")
print(f"  无 thinking: {(~has_thinking).sum()}")

print(f"\n  题型分布:")
for t in sorted(df['type'].unique()):
    n = (df['type'] == t).sum()
    think_n = has_thinking[df['type'] == t].sum()
    print(f"    {t:12s}: {n:5d} (thinking: {think_n})")

if has_thinking.any():
    lengths = df.loc[has_thinking, 'thinking'].str.len()
    print(f"\n  Thinking 长度: min={lengths.min()}, median={lengths.median():.0f}, ")
    print(f"                 mean={lengths.mean():.0f}, p95={lengths.quantile(0.95):.0f}, max={lengths.max()}")

In [ ]:
# =============================================
#  分词器 & 模型 (4-bit QLoRA)
# =============================================
# MODEL_PATH 已在上面通过 kagglehub 下载

# 4-bit 量化配置 (节省显存)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Tokenizer vocab size: {len(tokenizer)}")
print(f"pad_token: {tokenizer.pad_token} (id={tokenizer.pad_token_id})")

# 检查 special tokens
for tok in ['<think>', '</think>', '<|im_start|>', '<|im_end|>']:
    tid = tokenizer.convert_tokens_to_ids(tok)
    print(f"  {tok:20s} → token_id={tid}")

print(f"\n加载模型 (4-bit quantization)...")
t0 = time.time()
model_base = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)
print(f"模型加载完成: {time.time()-t0:.1f}s")
print(f"GPU memory: {torch.cuda.memory_allocated()/1024**3:.1f} GB")

## 2. Token 长度分析 — 检查截断

In [ ]:
# =============================================
#  Token 长度分析
# =============================================
PROMPT_SUFFIX = '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'
MAX_SEQ = 512  # 与 Kaggle notebook 一致

def build_thinking_text(row):
    """与 Kaggle notebook 完全一致的格式"""
    prompt = str(row['prompt'])
    answer = str(row['answer'])
    thinking = str(row.get('thinking', '')) if pd.notna(row.get('thinking', None)) else ''
    user_msg = prompt + PROMPT_SUFFIX
    
    if thinking.strip():
        text = (
            f"<|im_start|>user\n{user_msg}<|im_end|>\n"
            f"<|im_start|>assistant\n<think>\n{thinking.strip()}\n</think>\n\\boxed{{{answer}}}<|im_end|>"
        )
    else:
        text = (
            f"<|im_start|>user\n{user_msg}<|im_end|>\n"
            f"<|im_start|>assistant\n<think></think>\\boxed{{{answer}}}<|im_end|>"
        )
    return text

def build_answer_only_text(row):
    """V2 (0.68) 的格式: 空 think + boxed"""
    prompt = str(row['prompt'])
    answer = str(row['answer'])
    user_msg = prompt + PROMPT_SUFFIX
    return (
        f"<|im_start|>user\n{user_msg}<|im_end|>\n"
        f"<|im_start|>assistant\n<think></think>\\boxed{{{answer}}}<|im_end|>"
    )

# 计算 token 长度
think_lengths = []
ao_lengths = []
type_lengths = {t: [] for t in df['type'].unique()}

for _, row in df.iterrows():
    # Thinking format
    text_think = build_thinking_text(row)
    toks_think = tokenizer(text_think, add_special_tokens=False)['input_ids']
    think_lengths.append(len(toks_think))
    type_lengths[row['type']].append(len(toks_think))
    
    # Answer-only format (V2 style)
    text_ao = build_answer_only_text(row)
    toks_ao = tokenizer(text_ao, add_special_tokens=False)['input_ids']
    ao_lengths.append(len(toks_ao))

tl_think = np.array(think_lengths)
tl_ao = np.array(ao_lengths)

print(f"{'='*60}")
print(f"  TOKEN LENGTH ANALYSIS (current max_seq={MAX_SEQ})")
print(f"{'='*60}")
print(f"\n  Thinking format (9000 samples):")
print(f"    min={tl_think.min()}, median={np.median(tl_think):.0f}, mean={tl_think.mean():.0f}")
print(f"    p95={np.percentile(tl_think, 95):.0f}, p99={np.percentile(tl_think, 99):.0f}, max={tl_think.max()}")
print(f"    > {MAX_SEQ} tokens (dropped): {(tl_think > MAX_SEQ).sum()} ({(tl_think > MAX_SEQ).mean()*100:.1f}%)")
print(f"    > 1024 tokens: {(tl_think > 1024).sum()} ({(tl_think > 1024).mean()*100:.1f}%)")

print(f"\n  Answer-only format (V2 style):")
print(f"    min={tl_ao.min()}, median={np.median(tl_ao):.0f}, mean={tl_ao.mean():.0f}")
print(f"    p95={np.percentile(tl_ao, 95):.0f}, max={tl_ao.max()}")
print(f"    > {MAX_SEQ} tokens: {(tl_ao > MAX_SEQ).sum()} ({(tl_ao > MAX_SEQ).mean()*100:.1f}%)")

print(f"\n  Thinking 增加的 token 开销:")
overhead = tl_think - tl_ao
print(f"    平均每样本多 {overhead.mean():.0f} tokens (thinking chain)")
print(f"    占总长比例: {overhead.mean() / tl_think.mean() * 100:.1f}%")

print(f"\n  按题型 thinking token 长度:")
for t in sorted(type_lengths.keys()):
    arr = np.array(type_lengths[t])
    over_max = (arr > MAX_SEQ).sum()
    print(f"    {t:12s}: median={np.median(arr):.0f}, mean={arr.mean():.0f}, max={arr.max()}, >{MAX_SEQ}={over_max}")

## 3. 初始 Loss 诊断 — 训练前的 Loss 是多少？

比较 thinking vs answer-only 格式在 **未训练** 状态下的 loss, 确认 loss 是否本身就高

In [ ]:
# =============================================
#  初始 Loss 诊断 (不训练, 只跑 forward pass)
# =============================================

def compute_sample_loss(model, tokenizer, text, max_len=512):
    """计算单个样本的交叉熵 loss"""
    enc = tokenizer(text, return_tensors='pt', truncation=True, max_length=max_len)
    input_ids = enc['input_ids'].to(model.device)
    with torch.no_grad():
        out = model(input_ids=input_ids, labels=input_ids)
    return out.loss.item()


def compute_per_token_loss(model, tokenizer, text, max_len=512):
    """计算每个 token 的 loss, 返回 (token_losses, tokens)"""
    enc = tokenizer(text, return_tensors='pt', truncation=True, max_length=max_len)
    input_ids = enc['input_ids'].to(model.device)
    
    with torch.no_grad():
        out = model(input_ids=input_ids)
    
    # shift: predict next token
    logits = out.logits[:, :-1, :]
    targets = input_ids[:, 1:]
    losses = F.cross_entropy(logits[0], targets[0], reduction='none')
    tokens = [tokenizer.decode([t]) for t in input_ids[0][1:]]
    return losses.cpu().numpy(), tokens


# 随机采样 50 个样本做诊断
N_DIAG = 50
diag_df = df.sample(n=N_DIAG, random_state=42)

model_base.eval()
results = []

print(f"诊断 {N_DIAG} 个样本的初始 loss (base model, 无 LoRA)...")
for idx, (_, row) in enumerate(diag_df.iterrows()):
    text_think = build_thinking_text(row)
    text_ao = build_answer_only_text(row)
    
    loss_think = compute_sample_loss(model_base, tokenizer, text_think, max_len=MAX_SEQ)
    loss_ao = compute_sample_loss(model_base, tokenizer, text_ao, max_len=MAX_SEQ)
    
    # Token count
    n_tok_think = len(tokenizer(text_think, add_special_tokens=False)['input_ids'])
    n_tok_ao = len(tokenizer(text_ao, add_special_tokens=False)['input_ids'])
    
    results.append({
        'type': row['type'],
        'loss_thinking': loss_think,
        'loss_answer_only': loss_ao,
        'loss_diff': loss_think - loss_ao,
        'n_tokens_thinking': n_tok_think,
        'n_tokens_ao': n_tok_ao,
    })
    
    if idx < 3:
        print(f"  [{row['type']}] thinking_loss={loss_think:.3f}, ao_loss={loss_ao:.3f}, "
              f"diff={loss_think-loss_ao:+.3f}, tokens={n_tok_think}/{n_tok_ao}")

res_df = pd.DataFrame(results)

print(f"\n{'='*60}")
print(f"  初始 LOSS 诊断 (base model 无 LoRA, {N_DIAG} 样本)")
print(f"{'='*60}")
print(f"  Thinking format:    mean={res_df['loss_thinking'].mean():.4f}, std={res_df['loss_thinking'].std():.4f}")
print(f"  Answer-only format: mean={res_df['loss_answer_only'].mean():.4f}, std={res_df['loss_answer_only'].std():.4f}")
print(f"  Diff (think-ao):    mean={res_df['loss_diff'].mean():.4f}")
print(f"\n  → Thinking 格式让初始 loss 增加了 {res_df['loss_diff'].mean():.4f} ({res_df['loss_diff'].mean()/res_df['loss_answer_only'].mean()*100:.1f}%)")

print(f"\n  按题型初始 loss:")
for t in sorted(res_df['type'].unique()):
    t_df = res_df[res_df['type'] == t]
    print(f"    {t:12s}: thinking={t_df['loss_thinking'].mean():.4f}, ao={t_df['loss_answer_only'].mean():.4f}, diff={t_df['loss_diff'].mean():+.4f}")

In [ ]:
# =============================================
#  Per-token Loss 可视化 — 找出哪些 token 贡献最大 loss
# =============================================

# 对多个样本(每种题型各1个)做分段 loss 分析
print("=== 多样本分段 Loss 分析 ===\n")

# 每种类型取1个样本
seg_results = []
for qtype in sorted(diag_df['type'].unique()):
    sample_row = diag_df[diag_df['type'] == qtype].iloc[0]
    text = build_thinking_text(sample_row)
    losses, tokens = compute_per_token_loss(model_base, tokenizer, text, max_len=MAX_SEQ)
    
    # 找 <think> 和 </think> 分隔点
    think_start = None
    think_end = None
    for i, t in enumerate(tokens):
        if '<think>' in t and think_start is None:
            think_start = i
        if '</think>' in t and think_end is None:
            think_end = i
    
    if think_start is not None and think_end is not None:
        prompt_loss = losses[:think_start].mean() if think_start > 0 else 0
        think_loss = losses[think_start:think_end].mean() if think_end > think_start else 0
        answer_loss = losses[think_end:].mean() if think_end < len(losses) else 0
        
        prompt_n = think_start
        think_n = think_end - think_start
        answer_n = len(losses) - think_end
        
        seg_results.append({
            'type': qtype,
            'prompt_tokens': prompt_n, 'prompt_loss': prompt_loss,
            'think_tokens': think_n, 'think_loss': think_loss,
            'answer_tokens': answer_n, 'answer_loss': answer_loss,
            'total_tokens': len(losses),
        })
        
        print(f"  [{qtype:12s}] prompt={prompt_loss:.3f}({prompt_n}t)  "
              f"thinking={think_loss:.3f}({think_n}t)  "
              f"answer={answer_loss:.3f}({answer_n}t)  "
              f"think_pct={think_n/len(losses)*100:.0f}%")
    else:
        print(f"  [{qtype:12s}] ⚠️ 未找到 <think>/</ think> 分隔点")

# 汇总
if seg_results:
    seg_df = pd.DataFrame(seg_results)
    print(f"\n  --- 汇总 ---")
    print(f"  平均 prompt loss:   {seg_df['prompt_loss'].mean():.4f}")
    print(f"  平均 thinking loss: {seg_df['think_loss'].mean():.4f}  ← 这部分贡献最大!")
    print(f"  平均 answer loss:   {seg_df['answer_loss'].mean():.4f}")
    print(f"  thinking 占总 token 比例: {seg_df['think_tokens'].sum() / seg_df['total_tokens'].sum() * 100:.1f}%")

# 详细分析: 取 loss 最高的类型
print(f"\n\n=== 详细 Token Loss (最高 loss 类型) ===")
worst_type = seg_df.loc[seg_df['think_loss'].idxmax(), 'type'] if seg_results else diag_df['type'].iloc[0]
sample_row = diag_df[diag_df['type'] == worst_type].iloc[0]
text = build_thinking_text(sample_row)
losses, tokens = compute_per_token_loss(model_base, tokenizer, text, max_len=MAX_SEQ)

print(f"样本类型: {worst_type}")
print(f"总 tokens: {len(losses)}, 平均 loss: {losses.mean():.4f}")
print(f"\n--- Top 20 最高 loss 的 token ---")
top_idx = np.argsort(losses)[-20:][::-1]
for i in top_idx:
    print(f"  pos={i:4d}, loss={losses[i]:6.3f}, token='{tokens[i]}'")

## 4. 对比训练实验 — Thinking vs Answer-Only

用相同的 100 个样本, 分别用 thinking 和 answer-only 格式训练, 比较 loss 曲线

In [ ]:
# =============================================
#  LoRA setup helper
# =============================================
LORA_RANK = 32

def make_fresh_lora(base_model, alpha):
    """在干净的 base model 上创建新的 LoRA adapter"""
    lora_config = LoraConfig(
        r=LORA_RANK,
        lora_alpha=alpha,
        target_modules='all-linear',
        lora_dropout=0.05,
        bias='none',
        task_type=TaskType.CAUSAL_LM,
    )
    peft_model = get_peft_model(base_model, lora_config)
    peft_model.print_trainable_parameters()
    return peft_model

print(f"LoRA rank={LORA_RANK}, targets=all-linear")

In [ ]:
# =============================================
#  对比实验: 4种 (format × alpha) 组合
#  每轮删除旧 adapter → 重新 wrap → 训练 → 删除
# =============================================
from transformers import TrainerCallback
from peft import PeftModel

class LossLogger(TrainerCallback):
    """记录每步 loss"""
    def __init__(self):
        self.losses = []
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and 'loss' in logs:
            self.losses.append({'step': state.global_step, 'loss': logs['loss']})

# 只用 300 样本做快速对比 (全量太慢)
N_QUICK = 300
quick_df = df.sample(n=N_QUICK, random_state=42)

experiments = [
    ('thinking_alpha16', build_thinking_text, 16),
    ('thinking_alpha64', build_thinking_text, 64),
    ('answer_only_alpha16', build_answer_only_text, 16),
    ('answer_only_alpha64', build_answer_only_text, 64),
]

all_results = {}

for exp_name, builder, alpha in experiments:
    print(f"\n{'='*60}")
    print(f"  实验: {exp_name} (alpha={alpha})")
    print(f"{'='*60}")
    
    # Build dataset
    texts = quick_df.apply(builder, axis=1).tolist()
    hf_ds = Dataset.from_dict({'text': texts})
    
    # --- 关键: 每轮在干净 base_model 上重新创建 LoRA ---
    # model_base 是 quantized base, 不能 merge_and_unload (4-bit)
    # 所以用 peft 的 adapter 删除机制
    peft_model = make_fresh_lora(model_base, alpha)
    
    logger = LossLogger()
    
    trainer_args = SFTConfig(
        output_dir=f'/content/debug_{exp_name}',
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        num_train_epochs=1,
        learning_rate=2e-4,
        logging_steps=1,  # 每步都记录
        bf16=True,
        max_grad_norm=1.0,
        optim='adamw_torch',
        lr_scheduler_type='cosine',
        warmup_ratio=0.1,
        save_strategy='no',
        report_to='none',
        dataset_text_field='text',
        max_length=MAX_SEQ,
        packing=True,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={'use_reentrant': True},
    )
    
    trainer = SFTTrainer(
        model=peft_model,
        train_dataset=hf_ds,
        processing_class=tokenizer,
        args=trainer_args,
        callbacks=[logger],
    )
    
    t0 = time.time()
    result = trainer.train()
    elapsed = time.time() - t0
    
    all_results[exp_name] = {
        'final_loss': result.training_loss,
        'losses': logger.losses,
        'time': elapsed,
    }
    
    print(f"  Final loss: {result.training_loss:.4f}, Time: {elapsed:.0f}s")
    if logger.losses:
        print(f"  First loss: {logger.losses[0]['loss']:.4f}")
        print(f"  Last loss: {logger.losses[-1]['loss']:.4f}")
    
    # --- 关键: 彻底清理 trainer 和 peft model ---
    del trainer
    # 删除 adapter, 恢复 base model 到干净状态
    peft_model.delete_adapter('default')
    del peft_model
    gc.collect()
    torch.cuda.empty_cache()
    print(f"  GPU memory after cleanup: {torch.cuda.memory_allocated()/1024**3:.1f} GB")

print(f"\n{'='*60}")
print(f"  全部 {len(experiments)} 个实验完成!")
print(f"{'='*60}")

In [ ]:
# =============================================
#  结果汇总 & 可视化
# =============================================
import matplotlib.pyplot as plt

print(f"{'='*70}")
print(f"  📊 LOSS 对比汇总")
print(f"{'='*70}")
print(f"  {'实验':30s} {'初始 Loss':>10s} {'最终 Loss':>10s} {'降幅':>10s}")
print(f"  {'-'*62}")

for name, data in all_results.items():
    first = data['losses'][0]['loss'] if data['losses'] else 0
    final = data['final_loss']
    drop = first - final
    print(f"  {name:30s} {first:10.4f} {final:10.4f} {drop:+10.4f}")

# 画 loss 曲线
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: by format
for name, data in all_results.items():
    if data['losses']:
        steps = [d['step'] for d in data['losses']]
        losses = [d['loss'] for d in data['losses']]
        axes[0].plot(steps, losses, label=name, alpha=0.8)
axes[0].set_xlabel('Step')
axes[0].set_ylabel('Loss')
axes[0].set_title('Loss Curve Comparison')
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

# Right: bar chart of final loss
names = list(all_results.keys())
final_losses = [all_results[n]['final_loss'] for n in names]
colors = ['#2196F3' if 'thinking' in n else '#4CAF50' for n in names]
bars = axes[1].bar(range(len(names)), final_losses, color=colors)
axes[1].set_xticks(range(len(names)))
axes[1].set_xticklabels([n.replace('_', '\n') for n in names], fontsize=8)
axes[1].set_ylabel('Final Loss')
axes[1].set_title('Final Loss Comparison')
for bar, val in zip(bars, final_losses):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{val:.4f}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('/content/loss_comparison.png', dpi=150)
plt.show()
print('\n图表已保存到 /content/loss_comparison.png')

## 5. 关键诊断结论

根据以上实验数据, 分析 loss 异常大的原因

In [ ]:
# =============================================
#  诊断结论
# =============================================
print(f"{'='*70}")
print(f"  🔍 LOSS 诊断结论")
print(f"{'='*70}")

# 1. Thinking vs Answer-Only
think_16 = all_results.get('thinking_alpha16', {}).get('final_loss', 0)
ao_16 = all_results.get('answer_only_alpha16', {}).get('final_loss', 0)
think_64 = all_results.get('thinking_alpha64', {}).get('final_loss', 0)
ao_64 = all_results.get('answer_only_alpha64', {}).get('final_loss', 0)

print(f"\n📌 1. Thinking 格式 vs Answer-Only:")
if think_16 > 0 and ao_16 > 0:
    diff = think_16 - ao_16
    pct = diff / ao_16 * 100
    print(f"   alpha=16: thinking={think_16:.4f}, ao={ao_16:.4f}, diff={diff:+.4f} ({pct:+.1f}%)")
if think_64 > 0 and ao_64 > 0:
    diff = think_64 - ao_64
    pct = diff / ao_64 * 100
    print(f"   alpha=64: thinking={think_64:.4f}, ao={ao_64:.4f}, diff={diff:+.4f} ({pct:+.1f}%)")
print(f"   → Thinking 内容迫使模型学习程序化 CoT, 这些不是模型的原生输出分布")
print(f"   → 每个 thinking token 都贡献 cross-entropy loss, 而 thinking 占比~50% tokens")

print(f"\n📌 2. Alpha=64 vs Alpha=16:")
if think_16 > 0 and think_64 > 0:
    diff = think_64 - think_16
    print(f"   thinking: alpha16={think_16:.4f}, alpha64={think_64:.4f}, diff={diff:+.4f}")
if ao_16 > 0 and ao_64 > 0:
    diff = ao_64 - ao_16
    print(f"   ao:       alpha16={ao_16:.4f}, alpha64={ao_64:.4f}, diff={diff:+.4f}")
print(f"   → alpha/rank = 64/32 = 2.0 vs 16/32 = 0.5 → LoRA 更新强度差 4x")
print(f"   → 过强的 LoRA scaling 可能导致训练不稳定")

print(f"\n📌 3. 当前 notebook 的累积问题:")
print(f"   - 9000 样本 × 3 epochs = 27000 样本次训练 (V2 只有 600)")
print(f"   - alpha=64 → 4x 更强的每步更新")
print(f"   - Thinking 格式 → 更高的 per-sample loss")
print(f"   - 综合效应: 模型被 drive 远离基座分布 → 灾难性遗忘")

print(f"\n📌 建议修复方案:")
print(f"   A. 回退 alpha=16 (与 V2 一致)")
print(f"   B. 减少样本到 600-1000 (或只用 1 epoch)")
print(f"   C. 如果一定要用 thinking, 改用 boxed-only loss (只对 \\boxed{{}} 计算 loss)")
print(f"   D. 最安全: 用 V2 配置 (600样本, answer-only, alpha=16, 1 epoch)")

## 6. (可选) 完整训练 & 评测

选择最优配置运行完整训练

In [ ]:
# =============================================
#  OPTIONAL: 用最优配置训练完整模型
#  取消注释并运行
# =============================================

# BEST_FORMAT = 'answer_only'  # or 'thinking'
# BEST_ALPHA = 16
# BEST_N = 600
# BEST_EPOCHS = 1
# 
# best_df = df.sample(n=BEST_N, random_state=42)
# if BEST_FORMAT == 'answer_only':
#     texts = best_df.apply(build_answer_only_text, axis=1).tolist()
# else:
#     texts = best_df.apply(build_thinking_text, axis=1).tolist()
# 
# hf_ds = Dataset.from_dict({'text': texts})
# peft_model = setup_lora(model, BEST_ALPHA)
# 
# args = SFTConfig(
#     output_dir='/content/best_adapter',
#     per_device_train_batch_size=1,
#     gradient_accumulation_steps=4,
#     num_train_epochs=BEST_EPOCHS,
#     learning_rate=2e-4,
#     logging_steps=5,
#     bf16=True,
#     max_grad_norm=1.0,
#     optim='adamw_torch',
#     lr_scheduler_type='cosine',
#     warmup_ratio=0.1,
#     save_strategy='no',
#     report_to='none',
#     dataset_text_field='text',
#     max_length=1024,
#     packing=False,
#     gradient_checkpointing=True,
# )
# 
# trainer = SFTTrainer(
#     model=peft_model,
#     train_dataset=hf_ds,
#     processing_class=tokenizer,
#     args=args,
# )
# trainer.train()
# print(f'Training complete. Loss: {trainer.state.log_history[-1].get("train_loss", "?")}')